# LSTM Sentiment Analysis Project
Dataset: IMDB Movie Reviews (Kaggle)

This notebook builds a Sentiment Analysis model using **LSTM** with basic preprocessing.
No advanced NLP libraries are used.

## 1. Install Libraries

In [ ]:
!pip install tensorflow pandas numpy scikit-learn

## 2. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

## 3. Load Dataset
Download `IMDB Dataset.csv` from Kaggle and place it in the same folder.

In [ ]:
df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 4. Convert Labels to Numeric

In [ ]:
df['sentiment'].unique()

array(['positive', 'negative'], dtype=object)

In [ ]:
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


## 5. Train Test Split

In [ ]:
X = df['review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train


,review
39087,That's what I kept asking myself during the ma...
30893,I did not watch the entire movie. I could not ...
45278,A touching love story reminiscent of In the M...
16398,This latter-day Fulci schlocker is a totally a...
13653,"First of all, I firmly believe that Norwegian ..."
...,...
11284,`Shadow Magic' recaptures the joy and amazemen...
44732,I found this movie to be quite enjoyable and f...
38158,Avoid this one! It is a terrible movie. So wha...
860,This production was quite a surprise for me. I...


## 6. Tokenization

In [9]:
vocab_size = 10000

tokenizer = Tokenizer(num_words=vocab_size)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)


max_length = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

## 7. Padding Sequences

In [ ]:
# max_length = 200

# X_train_pad = pad_sequences(X_train_seq, maxlen=max_length)
# X_test_pad = pad_sequences(X_test_seq, maxlen=max_length)

## 8. Build LSTM Model

In [ ]:
model = Sequential()

model.add(Embedding(input_dim=vocab_size, output_dim=128, input_length=max_length))

model.add(LSTM(128))

model.add(Dropout(0.5))

model.add(Dense(1, activation='sigmoid'))



## 9. Compile Model

In [ ]:
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## 10. Train Model

In [ ]:
history = model.fit(
    X_train_pad,
    y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 257s 508ms/step - accuracy: 0.7330 - loss: 0.5024 - val_accuracy: 0.8716 - val_loss: 0.3130
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 258s 502ms/step - accuracy: 0.9025 - loss: 0.2485 - val_accuracy: 0.8792 - val_loss: 0.2931
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 501ms/step - accuracy: 0.9327 - loss: 0.1817 - val_accuracy: 0.8704 - val_loss: 0.3393
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 263s 503ms/step - accuracy: 0.9537 - loss: 0.1307 - val_accuracy: 0.8811 - val_loss: 0.3403
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 262s 503ms/step - accuracy: 0.9616 - loss: 0.1094 - val_accuracy: 0.8696 - val_loss: 0.3593


## 11. Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test_pad, y_test)
print('Test Accuracy:', accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 41s 128ms/step - accuracy: 0.8636 - loss: 0.3464
Test Accuracy: 0.8715999722480774


## 12. Predict Sentiment

In [ ]:
review = ['This movie was absolutely amazing']

seq = tokenizer.texts_to_sequences(review)

pad = pad_sequences(seq, maxlen=max_length)

prediction = model.predict(pad)

if prediction > 0.5:
    print('Positive Review')
else:
    print('Negative Review')

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 557ms/step
Positive Review


What is prediction here?

In your LSTM model, the last layer is usually:

Dense(1, activation='sigmoid')

✅ What sigmoid does

The sigmoid function converts output into a probability between 0 and 1:

👉 So your model output will be:

prediction = 0.0 → 1.0

🧠 Meaning of prediction

Value	Meaning

0.0	Definitely Negative

0.5	Uncertain

1.0	Definitely Positive

🔹 Why > 0.5 ?

if prediction > 0.5:

    print('Positive Review')

else:

    print('Negative Review')
✅ Reason: Decision Boundary

In binary classification, we need a cutoff (threshold) to decide:

Class 0 → Negative

Class 1 → Positive

👉 The default threshold is:

0.5
🧠 Interpretation:

If probability > 0.5 → more likely Positive

If probability ≤ 0.5 → more likely Negative